# Encoder-Only Transformer (Sentiment Classification)

A minimal BERT-style encoder that reads a whole review at once and predicts a
sentiment label (`positive` / `negative`). Everything is tiny on purpose:
small vocab, `d_model = 8`, one attention head, one encoder block.

## How it differs from the decoder-only model

| | Decoder-only (GPT-style) | Encoder-only (BERT-style) |
|---|---|---|
| Attention | **masked** — each token sees only the past | **bidirectional** — every token sees the whole sentence |
| Task | next-token prediction (per position) | sequence classification (one label per review) |
| Output head | `Linear(d_model, vocab_size)` at every position | `Linear(d_model, num_classes)` on a pooled vector |
| Special token | `<EOS>` separates prompt/answer | `<CLS>` at position 0 summarizes the sequence |

## Architecture

```mermaid
flowchart TD
    X["token ids<br/>(batch, seq)"] --> EMB["nn.Embedding(vocab, 8)<br/>(batch, seq, 8)"]
    EMB --> POS["PositionEncoding<br/>x + pe[:seq_len]<br/>(batch, seq, 8)"]

    subgraph BLOCK["EncoderBlock"]
        direction TB
        ATT["Bidirectional Self-Attention<br/>(no causal mask)"]
        N1["LayerNorm(x + attn)"]
        FF["Feed-Forward<br/>Linear 8-&gt;16, ReLU, Linear 16-&gt;8"]
        N2["LayerNorm(x + ff)"]
        ATT --> N1 --> FF --> N2
    end

    POS --> ATT
    POS -. residual .-> N1
    N1 -. residual .-> N2

    N2 --> POOL["take <CLS> vector<br/>x[:, 0]<br/>(batch, 8)"]
    POOL --> FC["nn.Linear(8, 2)<br/>logits (batch, 2)"]
    FC --> LOSS["CrossEntropyLoss<br/>vs labels (batch,)"]
```

## Shapes at every step

| Stage | Tensor | Shape |
|---|---|---|
| input | `X` | `(batch, seq)` |
| after embedding | word vectors | `(batch, seq, 8)` |
| after position encoding | positioned vectors | `(batch, seq, 8)` |
| attention scores | `Q @ K.T / sqrt(8)` | `(batch, seq, seq)` |
| after encoder block | contextualized vectors | `(batch, seq, 8)` |
| `<CLS>` pooled vector | `x[:, 0]` | `(batch, 8)` |
| after `fc` | logits | `(batch, 2)` — one score per class |


In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [28]:
# =====================================================
# Vocabulary
#
# <CLS> sits at the front of every review. After the encoder runs,
# its output vector is used as the summary of the whole sentence
# and fed to the classification head.
# <PAD> fills short reviews up to a common length.
# =====================================================
token_to_id = {
    "<CLS>": 0,
    "<PAD>": 1,
    "the": 2,
    "movie": 3,
    "was": 4,
    "great": 5,
    "terrible": 6,
    "i": 7,
    "loved": 8,
    "hated": 9,
    "it": 10,
    "a": 11,
    "waste": 12,
    "of": 13,
    "time": 14,
    "absolutely": 15,
    "amazing": 16,
    "boring": 17,
    "and": 18,
    "dull": 19,
    "best": 20,
    "ever": 21,
    "worst": 22,
    "not": 23
}

id_to_token = {v: k for k, v in token_to_id.items()}

# label ids
label_to_id = {"negative": 0, "positive": 1}
id_to_label = {v: k for k, v in label_to_id.items()}

vocab_size = len(token_to_id)
num_classes = len(label_to_id)
d_model = 8
max_len = 12

In [29]:
# =====================================================
# Training Reviews
#
# Each review starts with <CLS> and carries a sentiment label.
# Unlike the decoder model there is no next-token target -- the
# whole sentence maps to a single label.
# =====================================================
reviews = [
    (["<CLS>", "the", "movie", "was", "great"],                 "positive"),
    (["<CLS>", "i", "loved", "it"],                             "positive"),
    (["<CLS>", "absolutely", "amazing", "movie"],               "positive"),
    (["<CLS>", "the", "best", "movie", "ever"],                 "positive"),
    (["<CLS>", "i", "loved", "the", "movie"],                   "positive"),
    (["<CLS>", "the", "movie", "was", "terrible"],              "negative"),
    (["<CLS>", "i", "hated", "it"],                             "negative"),
    (["<CLS>", "a", "waste", "of", "time"],                     "negative"),
    (["<CLS>", "boring", "and", "dull"],                        "negative"),
    (["<CLS>", "the", "worst", "movie", "ever"],                "negative"),
    (["<CLS>", "the", "movie", "was", "not", "great"],          "negative"),
    (["<CLS>", "the", "movie", "was", "not", "amazing"],        "negative"),
]

In [30]:
# =====================================================
# Build Training Data
# X = padded token ids   (batch, max_len)
# Y = sentiment label     (batch,)
# =====================================================
def encode(words):
    ids = [token_to_id[w] for w in words]
    ids = ids[:max_len]
    ids = ids + [token_to_id["<PAD>"]] * (max_len - len(ids))
    return ids

X = torch.tensor([encode(words) for words, _ in reviews])
Y = torch.tensor([label_to_id[label] for _, label in reviews])

print("Input")
print(X)

print("\nLabels")
print(Y)

Input
tensor([[ 0,  2,  3,  4,  5,  1,  1,  1,  1,  1,  1,  1],
        [ 0,  7,  8, 10,  1,  1,  1,  1,  1,  1,  1,  1],
        [ 0, 15, 16,  3,  1,  1,  1,  1,  1,  1,  1,  1],
        [ 0,  2, 20,  3, 21,  1,  1,  1,  1,  1,  1,  1],
        [ 0,  7,  8,  2,  3,  1,  1,  1,  1,  1,  1,  1],
        [ 0,  2,  3,  4,  6,  1,  1,  1,  1,  1,  1,  1],
        [ 0,  7,  9, 10,  1,  1,  1,  1,  1,  1,  1,  1],
        [ 0, 11, 12, 13, 14,  1,  1,  1,  1,  1,  1,  1],
        [ 0, 17, 18, 19,  1,  1,  1,  1,  1,  1,  1,  1],
        [ 0,  2, 22,  3, 21,  1,  1,  1,  1,  1,  1,  1],
        [ 0,  2,  3,  4, 23,  5,  1,  1,  1,  1,  1,  1],
        [ 0,  2,  3,  4, 23, 16,  1,  1,  1,  1,  1,  1]])

Labels
tensor([1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0])


In [31]:
# =====================================================
# Positional Encoding
# (identical to the decoder model -- position still matters)
# =====================================================
class PositionEncoding(nn.Module):

    def __init__(self, d_model, max_len):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)

    def forward(self, word_embeddings):
        seq_len = word_embeddings.size(-2)
        return word_embeddings + self.pe[:seq_len]

In [32]:
# =====================================================
# Single Head Bidirectional Attention
#
# The key difference from the decoder: there is NO causal mask.
# Every token attends to every other token, so the <CLS> summary
# can look at the entire review at once.
#
# We still mask out <PAD> positions so padding never contributes
# to the attention weights.
# =====================================================
class Attention(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, pad_mask=None):

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(d_model)

        # pad_mask: (batch, seq) with True where token is <PAD>.
        # Expand to (batch, 1, seq) so those key positions are ignored.
        if pad_mask is not None:
            scores = scores.masked_fill(
                pad_mask.unsqueeze(1), -1e9
            )

        attention = F.softmax(scores, dim=-1)

        output = torch.matmul(attention, V)

        return output

In [33]:
# =====================================================
# Encoder Block
# (same shape as the decoder block, just bidirectional attention)
# =====================================================
class EncoderBlock(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        self.attention = Attention(d_model)

        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Linear(16, d_model),
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, pad_mask=None):

        attn = self.attention(x, pad_mask)

        x = self.norm1(x + attn)

        ff = self.ff(x)

        x = self.norm2(x + ff)

        return x

In [34]:
# =====================================================
# Encoder Only Transformer (Classifier)
#
# 1. embed + add position
# 2. run the encoder block (bidirectional)
# 3. take the <CLS> vector at position 0 as the sentence summary
# 4. project it to class logits
# =====================================================
class EncoderOnlyTransformer(nn.Module):

    def __init__(self, vocab_size, d_model, max_len, num_classes, pad_id):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.position = PositionEncoding(d_model, max_len)
        self.encoder = EncoderBlock(d_model)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        pad_mask = (x == self.pad_id)   # (batch, seq)

        x = self.embedding(x)
        x = self.position(x)
        x = self.encoder(x, pad_mask)

        cls = x[:, 0]                   # (batch, d_model) -- the <CLS> summary
        logits = self.fc(cls)           # (batch, num_classes)
        return logits

In [35]:
# =====================================================
# Create Model
# =====================================================
model = EncoderOnlyTransformer(
    vocab_size=vocab_size,
    d_model=d_model,
    max_len=max_len,
    num_classes=num_classes,
    pad_id=token_to_id["<PAD>"],
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
)

In [36]:
# =====================================================
# Training
# =====================================================
epochs = 500

for epoch in range(epochs):

    optimizer.zero_grad()

    logits = model(X)

    loss = criterion(logits, Y)

    loss.backward()

    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch:4d}  Loss = {loss.item():.4f}")

Epoch    0  Loss = 0.6988
Epoch   50  Loss = 0.0037
Epoch  100  Loss = 0.0004
Epoch  150  Loss = 0.0002
Epoch  200  Loss = 0.0002
Epoch  250  Loss = 0.0001
Epoch  300  Loss = 0.0001
Epoch  350  Loss = 0.0001
Epoch  400  Loss = 0.0001
Epoch  450  Loss = 0.0001


In [37]:
# =====================================================
# Predictions on the training set
# =====================================================
print("Predictions on training reviews\n")

model.eval()

with torch.no_grad():
    logits = model(X)
    preds = logits.argmax(dim=-1)

for (words, true_label), pred in zip(reviews, preds):
    text = " ".join(w for w in words if w != "<CLS>")
    mark = "OK " if id_to_label[pred.item()] == true_label else "XX "
    print(f"{mark}{text:35s} -> {id_to_label[pred.item()]:8s} (true: {true_label})")

Predictions on training reviews

OK the movie was great                 -> positive (true: positive)
OK i loved it                          -> positive (true: positive)
OK absolutely amazing movie            -> positive (true: positive)
OK the best movie ever                 -> positive (true: positive)
OK i loved the movie                   -> positive (true: positive)
OK the movie was terrible              -> negative (true: negative)
OK i hated it                          -> negative (true: negative)
OK a waste of time                     -> negative (true: negative)
OK boring and dull                     -> negative (true: negative)
OK the worst movie ever                -> negative (true: negative)
OK the movie was not great             -> negative (true: negative)
OK the movie was not amazing           -> negative (true: negative)


In [38]:
# =====================================================
# Inference on new (unseen) reviews
#
# Words must be in the vocabulary. The model generalizes by
# combining words it saw during training in new orders.
# =====================================================
def predict_sentiment(text):
    words = ["<CLS>"] + text.lower().split()
    ids = torch.tensor([encode(words)])

    with torch.no_grad():
        logits = model(ids)
        probs = F.softmax(logits, dim=-1)[0]
        pred = probs.argmax().item()

    print(f"Review    : {text}")
    print(f"Sentiment : {id_to_label[pred]}  "
          f"(neg={probs[0]:.2f}, pos={probs[1]:.2f})\n")

predict_sentiment("the movie was not great")
predict_sentiment("the movie was not amazing")
predict_sentiment("i hated the boring movie")
predict_sentiment("absolutely the best")
predict_sentiment("a terrible waste of time")

Review    : the movie was not great
Sentiment : negative  (neg=1.00, pos=0.00)

Review    : the movie was not amazing
Sentiment : negative  (neg=1.00, pos=0.00)

Review    : i hated the boring movie
Sentiment : negative  (neg=1.00, pos=0.00)

Review    : absolutely the best
Sentiment : positive  (neg=0.00, pos=1.00)

Review    : a terrible waste of time
Sentiment : negative  (neg=1.00, pos=0.00)

